<a href="https://colab.research.google.com/github/KyryloPerov/python_for_DS_HomeTasks/blob/main/HW_2_2_4_Unbalanced_multiclass_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

У цьому ДЗ ми потренуємось розв'язувати задачу багатокласової класифікації за допомогою логістичної регресії з використанням стратегій One-vs-Rest та One-vs-One, оцінити якість моделей та порівняти стратегії.

### Опис задачі і даних

**Контекст**

В цьому ДЗ ми працюємо з даними про сегментацію клієнтів.

Сегментація клієнтів – це практика поділу бази клієнтів на групи індивідів, які схожі між собою за певними критеріями, що мають значення для маркетингу, такими як вік, стать, інтереси та звички у витратах.

Компанії, які використовують сегментацію клієнтів, виходять з того, що кожен клієнт є унікальним і що їхні маркетингові зусилля будуть більш ефективними, якщо вони орієнтуватимуться на конкретні, менші групи зі зверненнями, які ці споживачі вважатимуть доречними та які спонукатимуть їх до купівлі. Компанії також сподіваються отримати глибше розуміння уподобань та потреб своїх клієнтів з метою виявлення того, що кожен сегмент цінує найбільше, щоб точніше адаптувати маркетингові матеріали до цього сегменту.

**Зміст**.

Автомобільна компанія планує вийти на нові ринки зі своїми існуючими продуктами (P1, P2, P3, P4 і P5). Після інтенсивного маркетингового дослідження вони дійшли висновку, що поведінка нового ринку схожа на їхній існуючий ринок.

На своєму існуючому ринку команда з продажу класифікувала всіх клієнтів на 4 сегменти (A, B, C, D). Потім вони здійснювали сегментовані звернення та комунікацію з різними сегментами клієнтів. Ця стратегія працювала для них надзвичайно добре. Вони планують використати ту саму стратегію на нових ринках і визначили 2627 нових потенційних клієнтів.

Ви маєте допомогти менеджеру передбачити правильну групу для нових клієнтів.

В цьому ДЗ використовуємо дані `customer_segmentation_train.csv`[скачати дані](https://drive.google.com/file/d/1VU1y2EwaHkVfr5RZ1U4MPWjeflAusK3w/view?usp=sharing). Це `train.csv`з цього [змагання](https://www.kaggle.com/datasets/abisheksudarshan/customer-segmentation/data?select=train.csv)

**Завдання 1.** Завантажте та підготуйте датасет до аналізу. Виконайте обробку пропущених значень та необхідне кодування категоріальних ознак. Розбийте на тренувальну і тестувальну вибірку, де в тесті 20%. Памʼятаємо, що весь препроцесинг ліпше все ж тренувати на тренувальній вибірці і на тестувальній лише використовувати вже натреновані трансформери.
Але в даному випадку оскільки значень в категоріях небагато, можна зробити обробку і на оригінальних даних, а потім розбити - це простіше. Можна також реалізувати процесинг і тренування моделі з пайплайнами. Обирайте як вам зручніше.

In [ ]:
!pip install -U imbalanced-learn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import classification_report

from imblearn.over_sampling import SMOTE, SMOTENC
from imblearn.combine import SMOTETomek
from imblearn.pipeline import Pipeline as ImbLearnPipeline

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
def value_counts_of_column(df, column, n=5):
  print(f"______Підрахунок значень колонки {column}______\n")

  value_counts = df[column].value_counts().sort_values(ascending=False).head(n)
  value_counts_norm = df[column].value_counts(normalize=True).sort_values(ascending=False).head(n) * 100

  print(f'Топ {n} значень:\n')
  print(f"{'Значення':<10} | {'Кількість':>10} | {'Відсоток':>8}")
  print('-' * 36)
  for value in value_counts.index:
    print(f"{str(value):<10} | {str(value_counts[value]):>10} | {value_counts_norm[value]:>7.2f}%")


def describe_columns_summary(df, top_n=5):
    print(f"{'column':<18} | {'dtype':<10} | {'unique val':<10} | {'missing':<8} | {'examples'}")
    print('-' * 90)

    for col in df.columns:
        dtype = df[col].dtype
        unique_vals = df[col].dropna().unique()
        n_missing = df[col].isna().sum()
        examples = ', '.join([str(val) for val in unique_vals[:top_n]])
        print(f"{col:<18} | {str(dtype):<10} | {len(unique_vals):<10} | {n_missing:<8} | [{examples}]")

    object_columns = df.select_dtypes(include='object').columns
    numeric_columns = df.select_dtypes(include='number').columns

    print('-' * 90)
    print(f"Numeric columns: {len(numeric_columns)}     Categorical columns: {len(object_columns)}")

In [ ]:
raw_df = pd.read_csv("/content/drive/MyDrive/ML/MLFH 3.0/data/Unbalanced_multiclass_classification/customer_segmentation_train.csv", index_col=0)
raw_df.head()

,Gender,Ever_Married,Age,Graduated,Profession,Work_Experience,Spending_Score,Family_Size,Var_1,Segmentation
ID,,,,,,,,,,
462809,Male,No,22,No,Healthcare,1.0,Low,4.0,Cat_4,D
462643,Female,Yes,38,Yes,Engineer,NaN,Average,3.0,Cat_4,A
466315,Female,Yes,67,Yes,Engineer,1.0,Low,1.0,Cat_6,B
461735,Male,Yes,67,Yes,Lawyer,0.0,High,2.0,Cat_6,B
462669,Female,Yes,40,Yes,Entertainment,NaN,High,6.0,Cat_6,A


In [ ]:
raw_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 8068 entries, 462809 to 461879
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Gender           8068 non-null   object 
 1   Ever_Married     7928 non-null   object 
 2   Age              8068 non-null   int64  
 3   Graduated        7990 non-null   object 
 4   Profession       7944 non-null   object 
 5   Work_Experience  7239 non-null   float64
 6   Spending_Score   8068 non-null   object 
 7   Family_Size      7733 non-null   float64
 8   Var_1            7992 non-null   object 
 9   Segmentation     8068 non-null   object 
dtypes: float64(2), int64(1), object(7)
memory usage: 693.3+ KB


In [ ]:
describe_columns_summary(raw_df)

column             | dtype      | unique val | missing  | examples
------------------------------------------------------------------------------------------
Gender             | object     | 2          | 0        | [Male, Female]
Ever_Married       | object     | 2          | 140      | [No, Yes]
Age                | int64      | 67         | 0        | [22, 38, 67, 40, 56]
Graduated          | object     | 2          | 78       | [No, Yes]
Profession         | object     | 9          | 124      | [Healthcare, Engineer, Lawyer, Entertainment, Artist]
Work_Experience    | float64    | 15         | 829      | [1.0, 0.0, 4.0, 9.0, 12.0]
Spending_Score     | object     | 3          | 0        | [Low, Average, High]
Family_Size        | float64    | 9          | 335      | [4.0, 3.0, 1.0, 2.0, 6.0]
Var_1              | object     | 7          | 76       | [Cat_4, Cat_6, Cat_7, Cat_3, Cat_1]
Segmentation       | object     | 4          | 0        | [D, A, B, C]
-----------------------------

In [ ]:
value_counts_of_column(raw_df, 'Segmentation')

______Підрахунок значень колонки Segmentation______

Топ 5 значень:

Значення   |  Кількість | Відсоток
------------------------------------
D          |       2268 |   28.11%
A          |       1972 |   24.44%
C          |       1970 |   24.42%
B          |       1858 |   23.03%


In [ ]:
train_df, val_df = train_test_split(raw_df, test_size=0.2, random_state=42, stratify=raw_df['Segmentation'])

input_cols = list(train_df.columns)[:-1]
target_col = 'Segmentation'
train_inputs, train_targets = train_df[input_cols].copy(), train_df[target_col].copy()
val_inputs, val_targets = val_df[input_cols].copy(), val_df[target_col].copy()

numeric_cols = train_inputs.select_dtypes(include=np.number).columns.tolist()
categorical_cols = train_inputs.select_dtypes('object').columns.tolist()

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(sparse_output=False, handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

In [ ]:
target_le = LabelEncoder()

train_targets_enc = target_le.fit_transform(train_targets)
val_targets_enc = target_le.transform(val_targets)

train_inputs_proc = preprocessor.fit_transform(train_inputs)
val_inputs_proc = preprocessor.transform(val_inputs)

Перевіримо чи всі пропущенні значення заповнені

In [ ]:
nan_in_train_proc = np.isnan(train_inputs_proc).sum()
nan_in_val_proc = np.isnan(val_inputs_proc).sum()

print(f"NaN before preprocessing in Train: {train_inputs.isnull().sum().sum()} After: {nan_in_train_proc}")
print(f"NaN before preprocessing in Val: {val_inputs.isnull().sum().sum()} After: {nan_in_val_proc}")

NaN before preprocessing in Train: 1246 After: 0
NaN before preprocessing in Val: 336 After: 0


Зберемо індекси для SMOTENC

In [ ]:
num_features_count = len(numeric_cols)
trained_onehot_encoder = preprocessor.named_transformers_['cat'].named_steps['onehot']
cat_features_encoded_count = len(trained_onehot_encoder.get_feature_names_out(categorical_cols))

categorical_features_indices_for_smotenc = list(range(num_features_count, num_features_count + cat_features_encoded_count))

In [ ]:
categorical_features_indices_for_smotenc

[3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 17,
 18,
 19,
 20,
 21,
 22,
 23,
 24,
 25,
 26,
 27]

**Завдання 2. Важливо уважно прочитати все формулювання цього завдання до кінця!**

Застосуйте методи ресемплингу даних SMOTE та SMOTE-Tomek з бібліотеки imbalanced-learn до тренувальної вибірки. В результаті у Вас має вийти 2 тренувальних набори: з апсемплингом зі SMOTE, та з ресамплингом з SMOTE-Tomek.

Увага! В нашому наборі даних є як категоріальні дані, так і звичайні числові. Базовий SMOTE не буде правильно працювати з категоріальними даними, але є його модифікація, яка буде. Тому в цього завдання є 2 виконання

  1. Застосувати SMOTE базовий лише на НЕкатегоріальних ознаках.

  2. Переглянути інформацію про метод [SMOTENC](https://imbalanced-learn.org/dev/references/generated/imblearn.over_sampling.SMOTENC.html#imblearn.over_sampling.SMOTENC) і використати цей метод в цій задачі. За цей спосіб буде +3 бали за це завдання і він рекомендований для виконання.

  **Підказка**: аби скористатись SMOTENC треба створити змінну, яка містить індекси ознак, які є категоріальними (їх номер серед колонок) і передати при ініціації екземпляра класу `SMOTENC(..., categorical_features=cat_feature_indeces)`.
  
  Ви також можете розглянути варіант використання варіації SMOTE, який працює ЛИШЕ з категоріальними ознаками [SMOTEN](https://imbalanced-learn.org/dev/references/generated/imblearn.over_sampling.SMOTEN.html)

In [ ]:
pipeline_orig = ImbLearnPipeline([
    ('preprocessor', preprocessor),
    ('classifier', OneVsRestClassifier(LogisticRegression(solver='liblinear', random_state=42)))
])

pipeline_smote = ImbLearnPipeline([
    ('preprocessor', preprocessor),
    ('smote', SMOTENC(categorical_features=categorical_features_indices_for_smotenc, random_state=42)),
    ('classifier', OneVsRestClassifier(LogisticRegression(solver='liblinear', random_state=42)))
])

pipeline_smote_tomek = ImbLearnPipeline([
    ('preprocessor', preprocessor),
    ('smote_tomek', SMOTETomek(
        smote=SMOTENC(categorical_features=categorical_features_indices_for_smotenc, random_state=42),
        random_state=42
    )),
    ('classifier', OneVsRestClassifier(LogisticRegression(solver='liblinear', random_state=42)))
])

**Завдання 3**.
  1. Навчіть модель логістичної регресії з використанням стратегії One-vs-Rest з логістичною регресією на оригінальних даних, збалансованих з SMOTE, збалансованих з Smote-Tomek.  
  2. Виміряйте якість кожної з натренованих моделей використовуючи `sklearn.metrics.classification_report`.
  3. Напишіть, яку метрику ви обрали для порівняння моделей.
  4. Яка модель найкраща?
  5. Якщо немає суттєвої різниці між моделями - напишіть свою гіпотезу, чому?

In [ ]:
print("--- Оцінка моделі на оригінальних даних ---")
pipeline_orig.fit(train_inputs, train_targets_enc)
targets_pred_orig = pipeline_orig.predict(val_inputs)
print(classification_report(val_targets_enc, targets_pred_orig, target_names=target_le.classes_))

print("\n--- Оцінка моделі на даних з SMOTENC ---")
pipeline_smote.fit(train_inputs, train_targets_enc)
targets_pred_smote = pipeline_smote.predict(val_inputs)
print(classification_report(val_targets_enc, targets_pred_smote, target_names=target_le.classes_))

print("\n--- Оцінка моделі на даних з SMOTE-Tomek ---")
pipeline_smote_tomek.fit(train_inputs, train_targets_enc)
targets_pred_smote_tomek = pipeline_smote_tomek.predict(val_inputs)
print(classification_report(val_targets_enc, targets_pred_smote_tomek, target_names=target_le.classes_))

--- Оцінка моделі на оригінальних даних ---
              precision    recall  f1-score   support

           A       0.42      0.45      0.43       394
           B       0.41      0.18      0.25       372
           C       0.49      0.61      0.54       394
           D       0.65      0.76      0.70       454

    accuracy                           0.51      1614
   macro avg       0.49      0.50      0.48      1614
weighted avg       0.50      0.51      0.49      1614


--- Оцінка моделі на даних з SMOTENC ---
              precision    recall  f1-score   support

           A       0.42      0.47      0.44       394
           B       0.40      0.24      0.30       372
           C       0.51      0.59      0.55       394
           D       0.67      0.72      0.69       454

    accuracy                           0.52      1614
   macro avg       0.50      0.51      0.50      1614
weighted avg       0.51      0.52      0.51      1614


--- Оцінка моделі на даних з SMOTE-Tomek --

**Обрана метрика для порівняння моделей**

Для порівняння моделей у цьому випадку я обрав **Macro Avg F1-score** (добре підходить для багатокласових задач з дисбалансом класів, оскільки враховує всі класи рівноцінно незалежно від їх розміру)

Також звертатиму увагу на recall для окремих класів, особливо для тих, де початково була низька продуктивність.

**Спостереження:**

- Original Model (без ресемплінгу): Показує найнижчі показники за всіма метриками, особливо низький recall для класу 'B' (0.18), що свідчить про те, що ця модель погано виявляє зразки цього класу.

- SMOTENC Model: Спостерігається невелике, але послідовне покращення за всіма метриками порівняно з оригінальною моделлю. Macro Avg F1-score зросла з 0.48 до 0.50, а recall для класу 'B' суттєво покращився з 0.18 до 0.24. Це вказує на те, що SMOTENC допоміг моделі краще навчитися розпізнавати менші класи.

- SMOTE-Tomek Model: Результати цієї моделі практично ідентичні результатам SMOTENC за всіма показниками. Це означає, що додавання кроку TomekLinks (який видаляє шумові та прикордонні зразки) після SMOTE не призвело до значних змін у продуктивності в даному випадку.

За результатами **SMOTENC та SMOTE-Tomek є трохи кращими** за модель без ресемплінгу. Між SMOTENC та SMOTE-Tomek немає суттєвої різниці за цими метриками.

На мою думку не бачимо суттєвої різниці, тому що дисбаланс класів був дуже помірним, і застосування технік ресамплінгу не призвело до значного покращення.

Хоча найменш представлений клас В ми таки навчились розпізнавати краще на resampled даних (його F1 score покращився з 0.25 до 0.30)